# Should we start gas now or wait?
At hour 5 the coal plant is 14 MW short of the required capacity. At hour 6 coal plus oil is still 20 MW short. Which starts now will cost less over the entire day? This notebook compares an eight-qubit Qiskit prototype with three classical decision rules and the organizers’ full-day solver. Run all cells from the extracted project folder.

In [1]:
from experiment import run
summary, hourly, trials, metrics, qc = run('results')
summary.round(2)

,method,day_cost,violations,startup_cost,gap_pct
0,Full-day MILP,327924.49,0,32930.0,0.00
1,Starter cheapest-first,330594.09,1,37950.0,0.81
2,One-hour least cost,351082.24,0,43360.0,7.06
3,Exact window search,331795.69,0,32930.0,1.18
4,QAOA 20 shots seed 0,337635.49,0,38080.0,2.96


### What actually switches on?
A 1 means the plant is online. Compare the one-hour rule, the sampled plan, and the exact choices.

In [2]:
hourly[hourly.hour.isin([5, 6])].reset_index(drop=True)

,method,hour,COAL,GAS_CC,PEAKER_A,PEAKER_B,OIL,demand,reserve
0,Full-day MILP,5,1,1,0,0,0,331.0,33.0
1,Full-day MILP,6,1,1,0,0,0,382.0,38.0
2,Starter cheapest-first,5,1,1,0,0,0,331.0,33.0
3,Starter cheapest-first,6,1,1,0,0,0,382.0,38.0
4,One-hour least cost,5,1,0,0,0,1,331.0,33.0
5,One-hour least cost,6,1,0,1,0,1,382.0,38.0
6,Exact window search,5,1,1,0,0,0,331.0,33.0
7,Exact window search,6,1,1,0,0,0,382.0,38.0
8,QAOA 20 shots seed 0,5,1,0,1,0,0,331.0,33.0
9,QAOA 20 shots seed 0,6,1,1,1,0,0,382.0,38.0


### How much does sampling help?
Each row averages 100 trials. The hit rate is the fraction finding the cheapest plan in the restricted window. The feasible share is the fraction of individual draws that yield a valid day.

In [3]:
trials.groupby(['shots','method'])[['hit_window_best','feasible_share']].mean().round(3)

hit_window_best  feasible_share
shots method                                  
5     QAOA                0.06           0.614
      Uniform             0.03           0.286
10    QAOA                0.10           0.636
      Uniform             0.04           0.273
20    QAOA                0.22           0.640
      Uniform             0.07           0.272
50    QAOA                0.46           0.634
      Uniform             0.17           0.282
200   QAOA                0.94           0.643
      Uniform             0.59           0.282

### What work happened outside the circuit?
We evaluated 256 candidates classically, tuned on that same table, and sampled from an ideal statevector. The full-day solver time is the organizer’s solver-only timer. Hardware queue time and noise are outside this experiment.

In [4]:
metrics.T

,0
valid_windows,72.000000
optimal_windows,1.000000
qaoa_best_probability,0.012350
uniform_best_probability,0.003906
gamma,1.178097
beta,-0.294524
classical_table_seconds,0.054422
tune_and_statevector_seconds,0.027382
milp_solver_seconds,0.017636


### Inspect the Qiskit circuit
The diagonal phase gate encodes the full cost table. Shown below is the logical circuit before decomposition into hardware gates.

In [5]:
print(qc.draw(output='text', fold=90))

     ┌───┐»
q_0: ┤ H ├»
     ├───┤»
q_1: ┤ H ├»
     ├───┤»
q_2: ┤ H ├»
     ├───┤»
q_3: ┤ H ├»
     ├───┤»
q_4: ┤ H ├»
     ├───┤»
q_5: ┤ H ├»
     ├───┤»
q_6: ┤ H ├»
     ├───┤»
q_7: ┤ H ├»
     └───┘»
«     ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

### What could we tell a judge?
On this day, committing gas earlier can avoid unnecessary starts. Compare the best two-hour decision with the hourly rule in the first table. The final row gives one preselected 20-sample QAOA run. The repeated-trial table shows how often the circuit recovers the best window decision. The classical table construction already reveals that answer. See README.md for the encoding, assumptions and a hardware follow-up.